# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:black; font-size:120%; text-align:left;padding:3.0px; background: #cceeff; border-bottom: 8px solid #004466" > TABLE OF CONTENTS<br><div>  
* [IMPORTS](#1)
* [INTRODUCTION](#2)
* [DATA PROCESSING](#3)
* [MODEL TRAINING](#4) 
* [MODEL INFERENCING](#5) 
* [OUTRO](#6)  
 

<a id="1"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #003380; border-bottom: 10px solid #80ffff"> PACKAGE IMPORTS<br><div> 

In [ ]:
%%time 

from IPython.display import display_html, clear_output, Markdown;
from gc import collect;

# Loading LightGBM and XGBoost latest versions:-
! pip install -q /kaggle/input/lightgbm410/lightgbm-4.1.0-py3-none-manylinux_2_28_x86_64.whl;
! pip install -q /kaggle/input/xgboost-2-0-0-whl/xgboost-2.0.0-py3-none-manylinux2014_x86_64.whl;

import xgboost as xgb;
import lightgbm as lgb;

clear_output();
print(f"\nXGBoost and LightGBM versions = {xgb.__version__} and {lgb.__version__}\n");
collect();

In [ ]:
%%time 

# General library imports:-
from itertools import combinations, product;
from copy import deepcopy;
import pandas as pd;
import numpy as np;
from holidays import country_holidays;
import polars as pl;
import polars.selectors as cs;

import joblib;
import os;
from os import system, getpid, walk, mkdir, path as ospath;
from psutil import Process;
import ctypes;
libc = ctypes.CDLL("libc.so.6");

from pprint import pprint;
from colorama import Fore, Style, init;
from warnings import filterwarnings;
filterwarnings('ignore');

from tqdm.notebook import tqdm;
import matplotlib.pyplot as plt;
import seaborn as sns;
%matplotlib inline

print();
collect();

In [ ]:
%%time 

# Model development:-
from sklearn.model_selection import (RepeatedStratifiedKFold as RSKF, 
                                     StratifiedKFold as SKF,
                                     KFold, 
                                     GroupKFold as GKF,
                                     RepeatedKFold as RKF, 
                                     cross_val_score);

from sklearn.pipeline import Pipeline, make_pipeline;
from sklearn.base import TransformerMixin, BaseEstimator;
from sklearn.preprocessing import MinMaxScaler;

from lightgbm import log_evaluation, early_stopping, LGBMRegressor as LGBMR;
from catboost import CatBoostRegressor as CBR;
from xgboost import XGBRegressor as XGBR;
from sklearn.ensemble import HistGradientBoostingRegressor as HGBR;
from sklearn.metrics import mean_absolute_error as mae, make_scorer;

print();
collect();

In [ ]:
%%time

# Defining global configurations and functions:-
sns.set({"axes.facecolor"       : "#ffffff",
         "figure.facecolor"     : "#ffffff",
         "axes.edgecolor"       : "#000000",
         "grid.color"           : "#ffffff",
         "font.family"          : ['Cambria'],
         "axes.labelcolor"      : "#000000",
         "xtick.color"          : "#000000",
         "ytick.color"          : "#000000",
         "grid.linewidth"       : 0.75,  
         "grid.linestyle"       : "--",
         "axes.titlecolor"      : '#0099e6',
         'axes.titlesize'       : 8.5,
         'axes.labelweight'     : "bold",
         'legend.fontsize'      : 7.0,
         'legend.title_fontsize': 7.0,
         'font.size'            : 7.5,
         'xtick.labelsize'      : 7.5,
         'ytick.labelsize'      : 7.5,        
        });

# Color printing    
def PrintColor(text:str, color = Fore.BLUE, style = Style.BRIGHT):
    "Prints color outputs using colorama using a text F-string";
    print(style + color + text + Style.RESET_ALL); 
    
def GetMemUsage(color = Fore.RED):
    """
    This function defines the memory usage across the kernel. 
    Source-
    https://stackoverflow.com/questions/61366458/how-to-find-memory-usage-of-kaggle-notebook
    """;
    
    pid = getpid();
    py = Process(pid);
    memory_use = py.memory_info()[0] / 2. ** 30;
    return PrintColor(f"---> RAM usage = {memory_use :.4} GB", color = color);

# Making sklearn pipeline outputs as dataframe:-
from sklearn import set_config; 
set_config(transform_output = "pandas");
pd.set_option('display.max_columns', 50);
pd.set_option('display.max_rows', 50);
pd.options.display.float_format = '{:,.2f}'.format;

# Setting global configurations for polars:-
pl.Config.activate_decimals(True).set_tbl_hide_column_data_types(True);
pl.Config(**dict(tbl_formatting = 'ASCII_FULL_CONDENSED',
                 tbl_hide_column_data_types = True,
                 tbl_hide_dataframe_shape = True,
                 fmt_float = "mixed",
                 tbl_cell_alignment = 'CENTER',
                 tbl_hide_dtype_separator = True,
                 tbl_cols = 100,
                 tbl_rows = 100,
                 fmt_str_lengths = 100,
                )
         );

print();
collect();

<a id="2"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #003380; border-bottom: 10px solid #80ffff"> INTRODUCTION<br><div> 

<div class="alert alert-block alert-info" style = "font-family: Cambria Math;font-size: 115%; color: black; background-color: #e6f9ff; border: dashed black 1.0px; padding: 3.5px" >
1. This notebook is my first tryst with the <b>Enefit </b> challenge. This is a tabular time series regression problem involving  energy prosumer data across 2021-2022. <b>Mean Absolute Error metric</b> is used here <br>
2. This notebook develops from my dataset and notebook for train data preparation <a> https://www.kaggle.com/code/ravi20076/enefit-traindataprep </a> <br>
3. We develop and infer models from this kernel. We use the same kernel and different versions to train and infer, using an interim dataset as an inferring mechanism <br>
    4. I use the kernel <a>https://www.kaggle.com/code/michaelo/enefit-polars-preprocessing </a> as a reference<br>
</div>

<a id="2.1"></a>
## <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size: 90%; text-align:left;padding:4.0px; background: maroon; border-bottom: 5px solid black"> VERSION DETAILS<br><div> 

| Version<br>Number | Version<br>Details | Preparation <br> date|LGBMR <br> CV| CBR <br> CV|XGBR <br> CV |HGBR <br> CV|Best LB <br>score|Single/<br> Ensemble|
| :-: | --- | :-: |  :-: |:-: |:-: |:-: |:-: |:-:|
|V1|* Features from public work <br> * ML tree models <br> * Weighted ensemble|07Nov2023 |41.82|45.74|47.23||92.93|Weighted <br> ensemble|
|V2|* Extra weather features <br> * ML tree models without early stopping<br> * Time series CV <br> * Weighted ensemble|08Nov2023 |78.56123|79.97057|85.46680|84.68283||Weighted <br> ensemble|

<a id="2.2"></a>
## <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size: 90%; text-align:left;padding:4.0px; background: maroon; border-bottom: 5px solid black"> CONFIGURATION PARAMETERS<br><div> 

| Parameter | Comments | Sample values|
| :-: | --- | :-: |
|version_nb | Version Number| integer value|
|test_req| Are we testing the code?| Y/N|
|test_frac| Test fraction for sampling and testing <br> Place small values for easy execution| float between 0 and 1|
|load_tr_data| Are we loading the train data here? <br> If we are inferring only, this is not required | Y/N|
|gpu_switch| Do we need a GPU here? |Y/N|
|state| Random seed| integer|
|target| Target column name| string value|
|dt_col| Date/ Datetime column name| string value|
|drop_cols| Dropped columns list| list|
|path_lbl| Data path for model training <br> I point this to my baseline data curation kernel| |
|test_path| Relevant path for test data| Competition artefacts|
|df_choice| Which data do I need for analysis? <br> Refer the baseline data prep kernel for details ||
|mdl_path| Path to dump trained models with joblib||
|inf_path| Appropriate path to extract the models for inference <br> I point to my baseline dataset with models trained as a starter||
|methods| All trained model methods, choose 1-more based on the memory constraints <br> For inferencing, all trained methods need to be present|list |
|ML| Do we need to do model training here? |Y/N |
|nbrnd_erly_stp| Number of early stopping rounds|integer value|
|early_stop_req| Do we want to use early stopping? |Y/N|
|refit_mdl| Do we want to refit the model on the entire train data? |Y/N|
|n_estm| Number of estimator trees |int|
|ens_weights| Weights if decided subjecively |dict|
|inference_req| Do we need to infer here? |Y/N|
|pstprcs_preds| Do we want to post-process predictions based on public work| Y/N|

In [ ]:
%%time 

# Configuration class:-
class CFG:
    """
    Configuration class for parameters and CV strategy for tuning and training
    Please use caps lock capital letters while filling in parameters
    """;
    
    # Data preparation:-   
    version_nb         = 2;
    test_req           = "N";
    test_frac          = 0.01;
    load_tr_data       = "N";
    gpu_switch         = "OFF"; 
    state              = 42;
    target             = 'target';
    dt_col             = "datetime";
    drop_cols          = [dt_col, "row_id", 'prediction_unit_id', 
                          'data_block_id', f"prediction_{dt_col}", f"origin_{dt_col}"];
    
    path_lbl           = f"/kaggle/input/enefittraindata";
    test_path          = f"/kaggle/input/predict-energy-behavior-of-prosumers/example_test_files/test.csv";
    df_choice          = f"XYtrain.parquet";
    mdl_path           = f'/kaggle/working/Models';
    inf_path           = f"/kaggle/input/enefitmodels";
     
    # Model Training:-
    methods            = ["LGBM1R", "LGBM3R"];
    ML                 = "N";
    cv_dt_cutoff       = f"2023-03-01";
    nbrnd_erly_stp     = 40;
    early_stop_req     = "N";
    refit_mdl          = "Y";   
    n_estm             = 450;
    
    # Ensemble:-    
    ens_weights        = {"LGBM1R": 0.05, "LGBM3R": 0.95};
    
    # Inference:-
    inference_req      = "Y";
    pstprcs_preds      = "Y";
    
    # Global variables for plotting:-
    grid_specs = {'visible': True, 'which': 'both', 'linestyle': '--', 
                  'color': 'lightgrey', 'linewidth': 0.75
                 };
    title_specs = {'fontsize': 9, 'fontweight': 'bold', 'color': 'tab:blue'};

print();
PrintColor(f"--> Configuration done!\n");
collect();

GetMemUsage();

In [ ]:
%%time 

# Defining the competition metric:-
def ScoreMetric(ytrue, ypred)-> float:
    """
    This function calculates the metric for the competition. 
    ytrue- ground truth array
    ypred- predictions
    returns - metric value (float)
    """;
    return mae(ytrue, ypred);

# Designing a custom scorer to use in cross_val_predict and cross_val_score:-
myscorer = make_scorer(ScoreMetric, greater_is_better = False, needs_proba=False,);

# Defining the post-processing function:-
def PostProcessPreds(preds: np.array, post_process = "Y"):
    """
    This is a post-processing function optionally used as per user's discretion
    """;
    return np.clip(preds, a_min = 0.0, a_max = 15000);

print();
collect();

# Defining the interim folder for models and other interim files:-
try:
    mkdir(CFG.mdl_path);
    PrintColor(f"Interim folder for models is done");
except:
    PrintColor(f"Interim folder for models is not done", color = Fore.RED);

GetMemUsage();

<a id="3"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #003380; border-bottom: 10px solid #80ffff"> DATA PROCESSING<br><div> 

In [ ]:
%%time 

if (CFG.load_tr_data == "Y" or CFG.ML == "Y") and CFG.test_req == "Y":
    if isinstance(CFG.test_frac, float):
        X = \
        pd.read_parquet(ospath.join(CFG.path_lbl, CFG.df_choice)).\
        groupby(CFG.dt_col).\
        sample(frac = CFG.test_frac).\
        dropna(subset = [CFG.target], axis=0);
    else:
        X = \
        pd.read_parquet(ospath.join(CFG.path_lbl, CFG.df_choice)).\
        groupby(['date_id']).\
        sample(n = CFG.test_frac).\
        dropna(subset = [CFG.target], axis=0);
    
    X    = X.loc[(~X[CFG.target].isna()) & (~(X["l2_production"] + X['l2_consumption']).isna())];
    X, y = X.drop(columns = [CFG.target], errors = "ignore"), X[CFG.target];
    PrintColor(f"---> Sampled train shapes for code testing = {X.shape} {y.shape}", 
               color = Fore.RED);
    X.index, y.index = range(len(X)), range(len(X));
    
    PrintColor(f"\n---> Train set columns for model development");
    with np.printoptions(linewidth = 130):
        pprint(X.columns);
    print();

elif CFG.load_tr_data == "Y" or CFG.ML == "Y":
    X = \
    pd.read_parquet(ospath.join(CFG.path_lbl, CFG.df_choice)).\
    dropna(subset = [CFG.target], axis=0);
    
    X    = X.loc[(~X[CFG.target].isna()) & (~(X["l2_production"] + X['l2_consumption']).isna())];
    X, y = X.drop(columns = [CFG.target], errors = "ignore"), X[CFG.target];
    X.index, y.index = range(len(X)), range(len(X));
    PrintColor(f"---> Train shapes = {X.shape} {y.shape}");

elif CFG.load_tr_data != "Y" or CFG.inference_req == "Y":
    PrintColor(f"---> Train data is not required as we are infering from the model");
    
print();
collect();
libc.malloc_trim(0);

GetMemUsage();

<a id="4"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #003380; border-bottom: 10px solid #80ffff"> MODEL TRAINING AND CV<br><div> 

In [ ]:
%%time 

# Initializing model I-O:-

if CFG.ML == "Y":  
    Mdl_Master = \
    {'CB1R': CBR(**{'task_type'           : "GPU" if CFG.gpu_switch == "ON" else "CPU",
                    'objective'           : "MAE",
                    'eval_metric'         : "MAE",
                    'bagging_temperature' : 0.25,
                    'colsample_bylevel'   : 0.5,
                    'iterations'          : CFG.n_estm,
                    'learning_rate'       : 0.045,
                    'max_depth'           : 7,
                    'l2_leaf_reg'         : 1.75,
                    'min_data_in_leaf'    : 2500,
                    'random_strength'     : 0.25, 
                    'verbose'             : 0,
                   }
                ), 

      'LGBM1R': LGBMR(**{'device'            : "gpu" if CFG.gpu_switch == "ON" else "cpu",
                         'objective'         : 'regression_l1',
                         'boosting_type'     : 'gbdt',
                         'random_state'      : CFG.state,
                         'colsample_bytree'  : 0.65,
                         'subsample'         : 0.65,
                         'learning_rate'     : 0.10,
                         'max_depth'         : 6,
                         'n_estimators'      : CFG.n_estm,
                         'num_leaves'        : 145,  
                         'reg_alpha'         : 0.01,
                         'reg_lambda'        : 1.25,
                         'verbosity'         : -1,
                        }
                    ),
     
      'LGBM2R': LGBMR(**{'device'              : "gpu" if CFG.gpu_switch == "ON" else "cpu",
                         'objective'           : 'regression',
                         'boosting_type'       : 'gbdt',
                         "data_sample_strategy": "goss",
                         'random_state'        : CFG.state,
                         'colsample_bytree'    : 0.7,
                         'subsample'           : 0.625,
                         'learning_rate'       : 0.0375,
                         'max_depth'           : 8,
                         'n_estimators'        : CFG.n_estm,
                         'num_leaves'          : 160,  
                         'reg_alpha'           : 0.001,
                         'reg_lambda'          : 0.001,
                         'verbosity'           : -1,
                        }
                    ),
     
        'LGBM3R': LGBMR(**{'device'            : "gpu" if CFG.gpu_switch == "ON" else "cpu",
                           'objective'         : 'regression_l1',
                           'boosting_type'     : 'gbdt',
                           'random_state'      : CFG.state,
                           'colsample_bytree'  : 0.84,
                           'learning_rate'     : 0.075,
                           'min_child_samples' : 125,
                           'n_estimators'      : CFG.n_estm,
                           'num_leaves'        : 140,
                           'reg_alpha'         : 0.155,
                           'reg_lambda'        : 0.16,
                           'subsample'         : 0.85,
                           'verbosity'         : -1,
                          }
                       ), 
     
        'LGBM4R': LGBMR(**{'device'            : "gpu" if CFG.gpu_switch == "ON" else "cpu",
                           'objective'         : 'regression_l1',
                           'boosting_type'     : 'gbdt',
                           'random_state'      : CFG.state,
                           'colsample_bytree'  : 0.4,
                           'learning_rate'     : 0.08,
                           'max_depth'         : 6,
                           'min_child_samples' : 120 ,
                           'n_estimators'      : CFG.n_estm,
                           'num_leaves'        : 90,
                           'reg_alpha'         : 0.80,
                           'reg_lambda'        : 0.001,
                           'subsample'         : 0.5,
                           'verbosity'         : -1,
                          }
                       ),
           
        "XGB1R" : XGBR(**{'tree_method'     : "gpu_hist" if CFG.gpu_switch == "ON" else "hist",
                          "objective"       : "reg:absoluteerror",
                          'max_depth'       : 8,
                          'random_state'    : CFG.state,
                          'n_estimators'    : CFG.n_estm,
                          'reg_alpha'       : 0.001, 
                          'reg_lambda'      : 0.85, 
                          'colsample_bytree': 0.5, 
                          'subsample'       : 0.6, 
                          'learning_rate'   : 0.045,
                          'verbosity'       : 0,
                         }
                      ),
         
        "XGB2R" : XGBR(**{'tree_method'     : "gpu_hist" if CFG.gpu_switch == "ON" else "hist",
                          "objective"       : "reg:absoluteerror",
                          'max_depth'       : 9,
                          'random_state'    : CFG.state,
                          'n_estimators'    : CFG.n_estm,
                          'reg_alpha'       : 0.85, 
                          'reg_lambda'      : 0.001, 
                          'colsample_bytree': 0.5, 
                          'subsample'       : 0.7, 
                          'learning_rate'   : 0.08,
                          'verbosity'       : 0,
                         }
                      ), 
     
        "HGBR" : HGBR(**{'random_state'      : CFG.state,
                         'learning_rate'     : 0.055,
                         'max_iter'          : 200,
                         'max_depth'         : 7,
                         'l2_regularization' : 2.5
                        }
                     ),
    };

print();
collect();

GetMemUsage();


In [ ]:
%%time 

if CFG.ML == "Y":
    # Initializing the models from configuration class:-
    methods = CFG.methods;

    # Initializing the model path for storage:-
    model_path = CFG.mdl_path;
           
    # Initializing other I-O:-
    Scores    = pd.DataFrame(index = methods, columns = ['TrainScore', "OOFScore"]);   
    FtreImp   = pd.DataFrame(index = X.iloc[0:10].\
                             drop(CFG.drop_cols, axis=1, errors = "ignore").\
                             columns, 
                             columns = [methods]
                            ).fillna(0);
      
print();
collect();
libc.malloc_trim(0);

GetMemUsage()

<div class="alert alert-block alert-info" style = "font-family: Cambria Math;font-size: 115%; color: black; background-color: #e6f9ff; border: dashed black 1.0px; padding: 3.5px" >
<b> Model training process:- </b> <br>
1. We will train models using the time series split CV strategy for the CV score. We will use the last 3 months of data as OOF <br>
2. We will store the model objects for later usage <br>
3. We will then infer from the model objects and predict the test data <br>
</div>

In [ ]:
%%time 

if CFG.ML == "Y":
    PrintColor(f"\n{'=' * 25} ML Training {'=' * 25}\n");
      
    # Initializing time-series based CV:-       
    Xtr  = X.loc[X[CFG.dt_col] < CFG.cv_dt_cutoff].drop(CFG.drop_cols, axis=1, errors = "ignore");   
    Xdev = X.loc[X[CFG.dt_col] >= CFG.cv_dt_cutoff].drop(CFG.drop_cols, axis=1, errors = "ignore");
    ytr  = y.loc[Xtr.index];
    ydev = y.loc[Xdev.index];   

    print(f"\n---> Shape => train = {Xtr.shape} {ytr.shape} | dev = {Xdev.shape} {ydev.shape}\n");
    if CFG.test_req == "Y":
        with np.printoptions(linewidth = 130): 
            print(Xtr.columns);
    
    # Fitting the models:- 
    for method in methods:
        model = Mdl_Master[method];

        if CFG.early_stop_req == "Y":
            if "LGBM" in method:
                model.fit(Xtr, ytr, 
                          eval_set = [(Xdev, ydev)], 
                          eval_metric = "mae",
                          callbacks = [log_evaluation(0,), 
                                       early_stopping(CFG.nbrnd_erly_stp, verbose = False)],
                         );

            elif "XGB" in method:
                model.fit(Xtr, ytr, 
                          eval_set = [(Xdev, ydev)], 
                          verbose = 0, 
                          eval_metric = "mae",
                         );  

            elif "CB" in method:
                model.fit(Xtr, ytr, 
                          eval_set = [(Xdev, ydev)], 
                          verbose = 0, 
                          early_stopping_rounds = CFG.nbrnd_erly_stp,
                          cat_features = ['county', "product_type"],
                         ); 
            else: 
                model.fit(Xtr, ytr);

        else:
            model.fit(Xtr, ytr);

        # Creating OOF scores:-
        tr_preds  = model.predict(Xtr);
        dev_preds = model.predict(Xdev);
        score     = ScoreMetric(ydev, PostProcessPreds(dev_preds, CFG.pstprcs_preds));
        tr_score  = ScoreMetric(ytr,  PostProcessPreds(tr_preds, CFG.pstprcs_preds));

        # Collating train and OOF scores:- 
        Scores.loc[method] = [tr_score, score];

        num_space  = 7- len(method);
        num_space1 = 2 if score >= 100 else 3;
        PrintColor(f"---> {method} {' '* num_space} OOF = {score:.2f} {' ' * num_space1}| Train = {tr_score:.2f}"); 

        # Collecting feature importances:-
        try:
            FtreImp[method] = \
            FtreImp[method].values.flatten() + model.feature_importances_;
        except:
            pass;
        
        if CFG.refit_mdl == "Y":
            PrintColor(f"---> Refitting the {method} model onto the complete train data", color = Fore.MAGENTA); 
            model.fit(X.drop(CFG.drop_cols, axis=1, errors = "ignore"), y);

        #  Saving the model for later usage:-
        joblib.dump(model, ospath.join(CFG.mdl_path, f'{method}V{CFG.version_nb}.model'));

    del Xtr, ytr, Xdev, ydev;
    collect();
    GetMemUsage();
    print();
    
    # Summarizing the model results across folds:-           
    PrintColor(f"\n---> Train-OOF scores across methods <---\n");
    Scores.index.name = "Method";
    display(Scores.style.format(precision = 5).\
            background_gradient(subset = Scores.columns, cmap = "icefire")
           );
    
    try: 
        FtreImp.to_csv(ospath.join(CFG.mdl_path, f"FtreImpV{CFG.version_nb}.csv"));
    except: 
        PrintColor(f"Feature importance was not exported\n");
                    
collect();
print();
libc.malloc_trim(0);
GetMemUsage();

In [ ]:
%%time 

if CFG.ML == "Y":
    if isinstance(FtreImp, pd.DataFrame) == True:
        FtreImp = make_pipeline(MinMaxScaler()).fit_transform(FtreImp);
        FtreImp.columns = CFG.methods;
    elif isinstance(FtreImp, pd.Series) == True:
        FtreImp = FtreImp.to_frame();
        FtreImp = make_pipeline(MinMaxScaler()).fit_transform(FtreImp);
        FtreImp.columns = CFG.methods;
    
    if len(CFG.methods) > 1:
        fig, axes = plt.subplots(len(CFG.methods), 1, sharex = True,
                                 figsize = (37, len(CFG.methods)* 8),
                                 gridspec_kw = {"hspace": 0.40, "wspace": 0.25},
                                );

        for i, method in enumerate(CFG.methods):
            ax = axes[i];
            FtreImp[method].plot.bar(color = "tab:blue", ax = ax);
            ax.set_title(f"{method} feature-importances", **CFG.title_specs);
            ax.set(xlabel = "", ylabel = "");
            ax.set_yticks(np.arange(0,1.01,0.05), labels = np.around(np.arange(0,1.01,0.05), 2), fontsize = 7);

        plt.tight_layout();
        plt.show();
    
    else:
        fig, ax = plt.subplots(1, 1, figsize = (32, 6.5));
        FtreImp[method].plot.bar(color = "tab:blue", ax = ax);
        ax.set_title(f"{method} feature-importances", **CFG.title_specs);
        ax.set(xlabel = "", ylabel = "");
        ax.set_yticks(np.arange(0,1.01,0.05), labels = np.around(np.arange(0,1.01,0.05), 2), fontsize = 7);
        
        plt.tight_layout();
        plt.show();  
     
collect();
print();
libc.malloc_trim(0);

<a id="5"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #003380; border-bottom: 10px solid #80ffff"> MODEL INFERENCING AND SUBMISSION<br><div> 

In [ ]:
%%time 

def MakeFtre(df, revealed_targets, client, electricity_prices, gas_prices, 
             historical_weather, forecast_weather, 
             train_test:str = "test", 
             **kwargs
            ):
    """
    This function makes secondary features for the provided datasets and returns the transformed data
    For the train data, lazy frame is returned
    For the test data, dataframe is returned
    Sources- 
    1. https://www.kaggle.com/code/michaelo/enefit-polars-preprocessing
    2. https://www.kaggle.com/code/vincentschuler/enefit-baseline-cross-validation
    
    Key note:-
    Revealed target table in the test data API is the same as the train set without the data block ID.
    This table is as on 2 days before the contemporary reporting date
    """;
    
    #  Placing a constant data block id column for test data and organizing the revealed targets table:-
    if train_test == "test":
        df = pl.from_pandas(df);
        df = df.\
        with_columns([pl.lit(1).alias("data_block_id").cast(pl.Int64),
                      pl.col("is_consumption").cast(pl.Int64),
                      pl.col("is_business").cast(pl.Int64)
                     ]
                    ).rename({f"prediction_{CFG.dt_col}": CFG.dt_col});
        
        client = \
        pl.from_pandas(client).\
        with_columns([pl.lit(1).alias("data_block_id").cast(pl.Int64),
                      pl.col("is_business").cast(pl.Int64)
                     ]
                    );
        gas_prices= \
        pl.from_pandas(gas_prices).\
        with_columns([pl.lit(1).\
                      alias("data_block_id").cast(pl.Int64)
                     ]
                    );
        electricity_prices = \
        pl.from_pandas(electricity_prices).\
        with_columns([pl.lit(1).\
                      alias("data_block_id").cast(pl.Int64)
                     ]
                    );
        historical_weather = \
        pl.from_pandas(historical_weather).\
        with_columns([pl.lit(1).alias("data_block_id").cast(pl.Int64)]
                    );
        forecast_weather = \
        pl.from_pandas(forecast_weather).\
        with_columns([pl.lit(1).\
                      alias("data_block_id")
                     ]);
        
        revealed_targets['is_consumption'] = \
        revealed_targets['is_consumption'].astype(np.int8);
        revealed_targets['data_block_id'] = 1;
        revealed_targets = \
        pl.from_pandas(revealed_targets).\
        with_columns([pl.lit(1).alias("data_block_id").cast(pl.Int64)]);
        
    else:
        pass;
    
    # Initiating trigonometric transforms and extra date features:-    
    trig_xforms = product(["day_nb", "weekday_nb", "month_nb", "hour_nb"], ["sin", "cos"], [1,2])
    trig_cols = \
    [getattr(pl.col(i) * pl.lit(k) * np.pi / pl.col(i).max(), j)().\
     alias(f"{j}_{i}_{k}").round(8).cast(pl.Float32) 
    for i, j, k in list(trig_xforms)
    ];
    
    df = \
    df.\
    with_columns([pl.col(CFG.dt_col).dt.ordinal_day().alias("day_nb").cast(pl.UInt16),
                  pl.col(CFG.dt_col).dt.weekday().alias("weekday_nb").cast(pl.UInt8),
                  pl.col(CFG.dt_col).dt.month().alias("month_nb").cast(pl.UInt8),
                  pl.col(CFG.dt_col).dt.hour().alias("hour_nb").cast(pl.UInt8),
                  pl.when(pl.col(CFG.dt_col).\
                          is_in(list(country_holidays(country = "EE", years = range(2019, 2025, 1)).keys()))
                         ).then(1).otherwise(0).alias("is_holiday").cast(pl.UInt8),
                 ]
                ).\
    with_columns(trig_cols);
    del trig_cols, trig_xforms;
    
    #  Placing lagged targets by consumption and production:- 
    if train_test == "train":
        l2_prod_csmp = \
        df.select(['prediction_unit_id', 'is_consumption', CFG.target, CFG.dt_col]).\
        group_by([CFG.dt_col, "prediction_unit_id"]).\
        agg(*[(pl.col(CFG.target).\
               filter(pl.col("is_consumption") == value).sum()
              ).alias("l2_production" if value == 0 else "l2_consumption") for value in [0,1]
             ]
           ).\
        with_columns([(pl.col("datetime") + pl.duration(days = 2))]);
        
    elif train_test == "test":
        l2_prod_csmp = \
        revealed_targets.select(['prediction_unit_id', 'is_consumption', CFG.target, CFG.dt_col]).\
        group_by([CFG.dt_col, "prediction_unit_id"]).\
        agg(*[(pl.col(CFG.target).\
               filter(pl.col("is_consumption") == value).sum()
              ).alias("l2_production" if value == 0 else "l2_consumption") for value in [0,1]
             ]
           ).\
        with_columns([(pl.col("datetime") + pl.duration(days = 2))]);
        
    # Processing gas prices and electricity prices:-
    gas_xform = \
    gas_prices.select('lowest_price_per_mwh', 'highest_price_per_mwh', 'data_block_id').\
                       with_columns((pl.col('lowest_price_per_mwh')/ 2 + pl.col('highest_price_per_mwh') / 2).\
                                    alias("mean_gas_price")
                                   );
    eprice_xform = \
    electricity_prices.select("euros_per_mwh", "data_block_id", "origin_date").\
    with_columns([(pl.col("origin_date") + pl.duration(days=2)).alias(CFG.dt_col)]).drop("origin_date");
    
    #  Transforming weather features:-
    hist_wthr_xform = \
    historical_weather.\
    with_columns((pl.col(CFG.dt_col) + pl.duration(days=1, hours= 13)).alias(CFG.dt_col)).\
    group_by([CFG.dt_col, 'data_block_id']).\
    agg([pl.exclude([CFG.dt_col, 'data_block_id']).mean().cast(pl.Float32).prefix("mean_"),
         pl.exclude([CFG.dt_col, 'data_block_id']).max().cast(pl.Float32).prefix("max_"),
         pl.exclude([CFG.dt_col, 'data_block_id']).min().cast(pl.Float32).prefix("min_")
        ]
       )
    
    fcst_wthr_xform = \
    forecast_weather.drop("longitude", "latitude").\
    group_by(['origin_datetime', 'hours_ahead', 'data_block_id', 'forecast_datetime']).\
    agg([pl.exclude(['origin_datetime', 'hours_ahead', 'data_block_id', 'forecast_datetime']).\
         mean().cast(pl.Float32).prefix("meanfcst_"),
         pl.exclude(['origin_datetime', 'hours_ahead', 'data_block_id', 'forecast_datetime']).\
         max().cast(pl.Float32).prefix("maxfcst_"),
         pl.exclude(['origin_datetime', 'hours_ahead', 'data_block_id', 'forecast_datetime']).\
         min().cast(pl.Float32).prefix("minfcst_")
        ]
       ).\
    with_columns([pl.col(f'forecast_{CFG.dt_col}').dt.convert_time_zone("Europe/Bucharest").\
                  alias(CFG.dt_col).dt.replace_time_zone(None),
                  pl.col("data_block_id").cast(pl.Int64)
                 ]
                ).drop(f'forecast_{CFG.dt_col}');
    
    # Joining the dataframes to make the final dataset:-
    df = \
    df.\
    join(client.drop("date"), how = "left", on = ["county", "is_business", "product_type", "data_block_id"]).\
    join(l2_prod_csmp, how = "left", on = [CFG.dt_col, "prediction_unit_id"]).\
    join(gas_xform, on = ["data_block_id"], how="left").\
    join(eprice_xform , on = ["data_block_id", CFG.dt_col], how="left").\
    join(hist_wthr_xform, on = ["data_block_id", CFG.dt_col], how="left").\
    join(fcst_wthr_xform, on = ["data_block_id", CFG.dt_col], how="left", suffix = "_fcst");
    
    collect();
    return df;

print();
collect();

In [ ]:
%%time

# Creating the testing environment:-
if CFG.inference_req == "Y":
    try: del X, y;
    except: pass;
      
    # Making the test environment for inferencing:-
    import enefit;
    try: 
        env       = enefit.make_env();
        iter_test = env.iter_test();
        PrintColor(f"\n---> Curating the inference environment");
    except: 
        pass;
    
    # Collating a list of models to be used for inferencing:-
    models = [];

    # Loading the models for inferencing:-
    if CFG.ML != "Y": 
        model_path = CFG.inf_path;
        PrintColor(f"---> Loading models from the input data for the kernel - V{CFG.version_nb}\n", 
                  color = Fore.RED);
    elif CFG.ML == "Y": 
        model_path = CFG.mdl_path;
        PrintColor(f"---> Loading models from the working directory for the kernel\n");
    
    # Loading the models:-
    mdl_lbl = [];
    models  = [];
    for _, _, filename in walk(model_path):
        mdl_lbl.extend(filename);
    for file in mdl_lbl:
        if file.endswith("model"):
            models.append(joblib.load(ospath.join(model_path, file)));
    
    mdl_lbl    = [m for m in mdl_lbl if m.endswith("model") == True];
    mdl_lbl    = [m.replace(r".model", "") for m in mdl_lbl];
    model_dict = {l:m for l,m in zip(mdl_lbl, models)};
    PrintColor(f"---> Trained models");    
    pprint(np.array(mdl_lbl), width = 100, indent = 10, depth = 1); 
    
print();
collect();  
libc.malloc_trim(0);
GetMemUsage();

In [ ]:
%%time 

if CFG.inference_req == "Y":
    print();
    counter = 0;
      
    for (test, revealed_targets, 
         client, historical_weather,
         forecast_weather, electricity_prices, 
         gas_prices, sample_prediction) in iter_test:
        
        if counter >= 99: num_space = 1;
        elif counter >= 9: num_space = 2;
        else: num_space = 3;
           
        PrintColor(f"{counter + 1}. {' ' * num_space} Inference", color = Fore.MAGENTA);
        Xtest = \
        MakeFtre(**dict(df                 = test, 
                        revealed_targets   = revealed_targets, 
                        client             = client, 
                        electricity_prices = electricity_prices, 
                        gas_prices         = gas_prices, 
                        historical_weather = historical_weather, 
                        forecast_weather   = forecast_weather,
                        train_test         = 'test'
                       )
                ).to_pandas().\
        sort_values(['row_id']).\
        drop(columns = CFG.drop_cols, errors = "ignore");
        
        if CFG.test_req == "Y": 
            print(Xtest.columns);
        else: 
            pass;
        del num_space;
        
        # Curating model predictions across methods and folds:-        
        preds = pd.DataFrame(columns = CFG.methods, index = test['row_id']).fillna(0);
        
        for method in list(CFG.ens_weights.keys()):
            for mdl_lbl, mdl in model_dict.items():
                if mdl_lbl.startswith(f"{method}"):
                    if CFG.test_req == "Y":
                        print(mdl_lbl, sep = " ", end = " |");
                    else:
                        pass;
                    test_preds    = PostProcessPreds(mdl.predict(Xtest),CFG.pstprcs_preds);
                    preds[method] = preds[method].values + test_preds;
        if CFG.test_req == "Y": print();
        else: pass;
        
        # Curating the weighted average model predictions:-       
        sample_prediction[CFG.target] = \
        np.average(preds.values, weights= list(CFG.ens_weights.values()), axis=1);
              
        try: 
            env.predict(sample_prediction);
        except: 
            PrintColor(f"---> Submission did not happen as we have the file already");
            pass;
        
        counter = counter+1;
        collect();
    
    PrintColor(f"\n---> Submission file\n");
    display(sample_prediction.head(10));
    
else:
    PrintColor(f"\n---> Inference is not required\n", color = Fore.RED);
            
print();
collect();  
libc.malloc_trim(0);
GetMemUsage(); 

<a id="6"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #003380; border-bottom: 10px solid #80ffff"> OUTRO<br><div> 

<div class="alert alert-block alert-info" style = "font-family: Calibri;font-size: 120%; color: black; background-color: #ffeecc; border: dashed black 1.0px; padding: 3.5px" >
<b> My baseline work is as below-</b> <br>
1. <b> Train Data preparation </b> - Uses the feature maker function using polars to develop train set features. This is pipeline compatible and saves time to build models quickly. Link is as below<br>
    <a> https://www.kaggle.com/code/ravi20076/enefit-traindataprep </a> <br>
2. <b> Train data with secondary features </b> - Output of the train data prep kernel is stored in this dataset and used as input here. Link is as below- <br>
   <a> https://www.kaggle.com/datasets/ravi20076/enefittraindata/ </a> <br>
3. <b> ML Model objects- </b> This dataset saves the trained models and uses them to infer here. Link is as below-<br>
    <a> https://www.kaggle.com/datasets/ravi20076/enefitmodels/ </a> <br>
</div>  